In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# NASDAQ DATA

# 
symbols = [
    "GOOGL","AVGO","MSFT","META","NVDA","AAPL","TSLA",
]



dfs = {}

for s in symbols:
    filepath = f"NASDAQ_DATA/{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── GOOGL ──────────────────────────
  Shape : (5295, 62)
  Head  :
              datetime ticker  open_price  high_price  low_price  close_price  \
0  2005-06-03 21:30:00  GOOGL    7.176706    7.239517   6.941975     7.013297   
1  2005-06-06 21:30:00  GOOGL    7.065850    7.350873   7.052586     7.280554   
2  2005-06-07 21:30:00  GOOGL    7.444714    7.497017   7.264541     7.335111   
3  2005-06-08 21:30:00  GOOGL    7.328354    7.336858   6.956742     6.995781   
4  2005-06-09 21:30:00  GOOGL    7.124904    7.219496   7.020805     7.164695   

         volume    EMA_10    EMA_20  EMA_ratio  ...  volume_lag_3  \
0  7.519087e+08  6.722983  6.354545   1.103666  ...  8.902334e+08   
1  9.031041e+08  6.824359  6.442736   1.130041  ...  1.408767e+09   
2  9.726738e+08  6.917223  6.527724   1.123686  ...  7.186961e+08   
3  1.031882e+09  6.931506  6.572301   1.064434  ...  7.519087e+08   
4  6.576569e+08  6.973904  6.628719   1.080857  ...  9.031041e+08   

   volume_lag_4  volume_lag_5 

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'NASDAQ_DATA/{symbol}_daily.csv', parse_dates=["datetime"])
    
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # Date-based split: train = up to 2018-12-31, test = 2019 onwards
    train_df = df[df['datetime'] <= '2018-12-31']
    test_df  = df[df['datetime'] >  '2018-12-31']

    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range   : {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Train rows: {len(train_df)}  Test rows: {len(test_df)}")


GOOGL
  Raw train UP%      : 51.47%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2005-06-13 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-18
  Train rows: 3169  Test rows: 1763

AVGO
  Raw train UP%      : 53.23%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2010-06-01 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-18
  Train rows: 1999  Test rows: 1783

MSFT
  Raw train UP%      : 49.74%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Train date range   : 1999-03-12 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-18
  Train rows: 4606  Test rows: 1783

META
  Raw train UP%      : 53.20%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Train date range   : 2013-03-15 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-18
  Train rows: 1361  Test rows: 1741

NVDA
  Raw train UP%      : 50.57%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.00

In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    # stationary price action (replaces raw price/volume levels)
    "ret_1", "gap", "intraday_range", "body",
    # trend (stationary)
    "EMA_ratio", "macd_norm",
    # momentum
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    # volatility (ratios, not levels)
    "ATR_ratio", "BB_pct", "realized_vol",
    # volume (stationary)
    "OBV_momentum", "volume_ratio", "volume_surge", "MFI_14",
    # lagged returns
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    # regime / persistence
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
    # market context (QQQ / SPY) — stationary, trailing, no look-ahead
    "qqq_ret_1", "qqq_mom_5", "qqq_mom_20", "spy_ret_1", "spy_mom_20", "rel_strength",
]

TARGET_COLS = ["price_pct_change_5d_vn"]


def _make_seqs(X, y, seq_len):
    """Simple sliding-window builder."""
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


# ── Market context (QQQ / SPY): built ONCE, merged per symbol ─────────────────
# All features are TRAILING (pct_change) -> value at date d uses only data <= d.
def _load_close(path):
    m = pd.read_csv(path, parse_dates=["datetime"])[["datetime", "close_price"]]
    return m.sort_values("datetime")

_qqq = _load_close("NASDAQ_DATA/QQQ_daily.csv").rename(columns={"close_price": "_qc"})
_spy = _load_close("NASDAQ_DATA/SPY_daily.csv").rename(columns={"close_price": "_sc"})
market = _qqq.merge(_spy, on="datetime", how="outer").sort_values("datetime").reset_index(drop=True)
market["qqq_ret_1"]  = market["_qc"].pct_change(1)
market["qqq_mom_5"]  = market["_qc"].pct_change(5)
market["qqq_mom_20"] = market["_qc"].pct_change(20)
market["spy_ret_1"]  = market["_sc"].pct_change(1)
market["spy_mom_20"] = market["_sc"].pct_change(20)
market = market[["datetime", "qqq_ret_1", "qqq_mom_5", "qqq_mom_20", "spy_ret_1", "spy_mom_20"]]
print(f"Market features built: {market['datetime'].min().date()} -> {market['datetime'].max().date()}")


for symbol in symbols:
    filepath = f"NASDAQ_DATA/{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    # merge market context on date (left join; values are trailing -> no look-ahead)
    df = df.merge(market, on="datetime", how="left")

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── Regime / persistence features ─────────────────────────────────────────
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── Stationary price-action features (regime-transferable) ────────────────
    df["ret_1"]          = df["close_price"].pct_change(1)
    df["gap"]            = df["open_price"] / df["close_price"].shift(1) - 1
    df["intraday_range"] = (df["high_price"] - df["low_price"]) / df["close_price"]
    df["body"]           = (df["close_price"] - df["open_price"]) / df["open_price"]
    df["macd_norm"]      = df["MACD_hist"] / df["close_price"]
    df["rel_strength"]   = df["ret_1"] - df["qqq_ret_1"]   # excess return vs QQQ (trailing)

    # ── 5-day forward close % change target ───────────────────────────────────
    df["price_pct_change_5d"] = (df["close_price"].shift(-5) / df["close_price"] - 1) * 100
    # trailing daily-return vol (no look-ahead), scaled to 5d horizon, in %, floored
    _daily_ret = df["close_price"].pct_change()
    df["vol_denom_5d"] = (_daily_ret.rolling(20).std() * np.sqrt(5) * 100).clip(lower=0.5)
    # target = 5d return in units of its own trailing 5d volatility (z-score)
    df["price_pct_change_5d_vn"] = df["price_pct_change_5d"] / df["vol_denom_5d"]

    df = df.dropna().reset_index(drop=True)

    # ── (Outlier clipping moved BELOW the split: TRAIN stats only, no test leakage) ──


    # ── 3-way split (no leakage): val carved from pre-2019; test = 2019+ touched once ──
    pre  = df[df['datetime'] <  '2019-01-01'].reset_index(drop=True)   # 2018-12-31 bar stays OUT of test
    post = df[df['datetime'] >= '2019-01-01']                        # test strictly 2019+
    cut  = int(len(pre) * 0.85)
    train_df = pre.iloc[:cut - 5]    # 5-row embargo: last train 5d targets must not reach val
    val_df   = pre.iloc[cut:-5]      # 5-row embargo: last val 5d targets must not reach test (2019)
    test_df  = post

    # ── Clip outliers on TRAIN ONLY (band from train); val/test left as-is (no leakage) ──
    for col, sigma in {"price_pct_change_5d_vn": 3}.items():
        mean, std = train_df[col].mean(), train_df[col].std()
        train_df  = train_df[train_df[col].between(mean - sigma * std, mean + sigma * std)]

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    val_X_raw   = val_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)
    val_y_raw   = val_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    val_X_scaled   = feature_scaler.transform(val_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    val_y_scaled   = target_scaler.transform(val_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        "val_X_scaled":   val_X_scaled,
        "val_y_scaled":   val_y_scaled,
        "test_vol_denom": test_df["vol_denom_5d"].values.astype(np.float32),
        "X_train": X_train,  "y_train": y_train,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Train date range: {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range: {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")

Market features built: 1993-11-11 -> 2026-06-22

── GOOGL ────────────────────────────────────────────────────────────
  Rows  → train:2838  val:505  test:1872
  Seqs  → train:2808  test:1842
  Shape → X:(2808, 30, 33)  y:(2808, 1)
  Train date range: 2005-07-01 → 2016-12-12
  Test  date range: 2019-01-02 → 2026-06-12
  Scalers saved → models/GOOGL/

── AVGO ────────────────────────────────────────────────────────────
  Rows  → train:1799  val:318  test:1872
  Seqs  → train:1769  test:1842
  Shape → X:(1769, 30, 33)  y:(1769, 1)
  Train date range: 2010-06-21 → 2017-09-11
  Test  date range: 2019-01-02 → 2026-06-12
  Scalers saved → models/AVGO/

── MSFT ────────────────────────────────────────────────────────────
  Rows  → train:3990  val:711  test:1872
  Seqs  → train:3960  test:1842
  Shape → X:(3960, 30, 33)  y:(3960, 1)
  Train date range: 2000-01-20 → 2016-02-19
  Test  date range: 2019-01-02 → 2026-06-12
  Scalers saved → models/MSFT/

── META ───────────────────────────────────

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Test: {len(test_loader)} batches")

    sample_X, sample_y, sample_w = next(iter(train_loader))
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 1)")
    print(f"  Sample w: {sample_w.shape} → (batch,)")


GOOGL — Train: 88 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 33]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

AVGO — Train: 56 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 33]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

MSFT — Train: 124 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 33]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

META — Train: 37 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 33]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

NVDA — Train: 124 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 33]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

AAPL — Train: 12

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}

# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 1)
        self.skip_fc        = nn.Linear(hidden_size * 2, 1)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss ────────────────────────────────────────────────────────────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets, weights=None):
        # Plain SmoothL1 (Huber, delta=1.0) on the standardized target.
        # Anti-collapse stack removed (BCE / std_match / collapse / sign):
        # it drove the mean-drift collapse. alpha/beta/gamma/weights kept in
        # the signature only for call-site compatibility; unused.
        return nn.SmoothL1Loss(beta=1.0)(preds, targets)


# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor
    weight_map[3] *= oversample_factor
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values ──────────────────────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val", "test"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        if split == "test" and "test_vol_denom" in data:
            data["test_vol_denom"] = data["test_vol_denom"][mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


Using device: cuda

Cleaning NaN values from all symbols...
  GOOGL train: 2838 → 2838 rows
  GOOGL val: 505 → 505 rows
  GOOGL test: 1872 → 1872 rows
  AVGO train: 1799 → 1799 rows
  AVGO val: 318 → 318 rows
  AVGO test: 1872 → 1872 rows
  MSFT train: 3990 → 3990 rows
  MSFT val: 711 → 711 rows
  MSFT test: 1872 → 1872 rows
  META train: 1214 → 1214 rows
  META val: 212 → 212 rows
  META test: 1872 → 1872 rows
  NVDA train: 3991 → 3991 rows
  NVDA val: 711 → 711 rows
  NVDA test: 1872 → 1872 rows
  AAPL train: 4003 → 4003 rows
  AAPL val: 711 → 711 rows
  AAPL test: 1872 → 1872 rows
  TSLA train: 1610 → 1610 rows
  TSLA val: 284 → 284 rows
  TSLA test: 1872 → 1872 rows


In [7]:
# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["val_X_scaled"]
    val_y      = data["val_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers     = trial.suggest_int("num_layers", 2, 3)
        dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical("batch_size", [32, 64])
        seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha          = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta           = trial.suggest_float("beta", 0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.5, 2.0, step=0.1)
        gamma          = trial.suggest_float("gamma", 0.5, 3.0, step=0.5)

        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            shuffle=True,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        SEEDS       = [42, 123, 456]
        seed_scores = []

        for seed_idx, seed in enumerate(SEEDS):
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = StockPctChangeBiLSTMAttention(
                input_size  = input_size,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                dropout     = dropout
            ).to(device)

            criterion = JointPredictionLoss(alpha=alpha, beta=beta, gamma=gamma)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

            EPOCHS         = 50
            PATIENCE       = 10
            best_val_score = 0.0
            patience_ctr   = 0
            seed_collapsed = False

            for epoch in range(1, EPOCHS + 1):
                model.train()
                for X_batch, y_batch, w_batch in train_ld:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    w_batch = w_batch.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch, w_batch)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                all_preds, all_actuals = [], []
                with torch.no_grad():
                    for X_batch, y_batch, _ in val_ld:
                        all_preds.append(model(X_batch.to(device)).cpu().numpy())
                        all_actuals.append(y_batch.numpy())

                if len(all_preds) == 0:
                    seed_collapsed = True
                    break

                val_preds   = np.vstack(all_preds)
                val_actuals = np.vstack(all_actuals)

                price_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 0]).astype(int),
                    np.sign(val_preds[:, 0]).astype(int)
                )

                price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])

                pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]), 1e-6)
                pred_up_pct    = (val_preds[:, 0] > 0).mean()
                actual_up_pct  = (val_actuals[:, 0] > 0).mean()
                bias_gap       = abs(pred_up_pct - actual_up_pct)

                if epoch >= 5:
                    if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                        seed_collapsed = True
                        break

                balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

                val_score = (
                    0.60 * price_dir +
                    0.40 * max(price_corr, 0.0) -
                    abs(1.0 - pred_std_ratio) * 0.3 -
                    bias_gap * 3.0 -
                    balance_penalty
                )

                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_ctr   = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= PATIENCE:
                        break

            seed_scores.append(-1.0 if seed_collapsed else best_val_score)

            trial.report(float(np.mean(seed_scores)), seed_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(seed_scores))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    # ── Save best params for reuse (skip Optuna when only the dataset changes) ─
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_best_params.json", "w") as f:
        json.dump({"best_params": study.best_params,
                   "best_value":  float(study.best_value)}, f, indent=2)
    print(f"  Best params saved -> models/{symbol}/{symbol}_best_params.json")

    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

[I 2026-06-24 13:44:25,551] A new study created in memory with name: bilstm_attention_GOOGL



═══════════════════════════════════════════════════════
  Optuna search — GOOGL
═══════════════════════════════════════════════════════


[I 2026-06-24 13:45:01,209] Trial 0 finished with value: 0.022914910010480117 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: 0.022914910010480117.
[I 2026-06-24 13:45:42,253] Trial 1 finished with value: 0.08122238092746391 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 1 with value: 0.08122238092746391.
[I 2026-06-24 13:46:29,405] Trial 2 finished with value: 0.14711368602959723 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 1.0}. Best is trial 2

  Best params saved -> models/GOOGL/GOOGL_best_params.json

  Best joint score : 0.1849
  Best trial       : 42
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.30000000000000004
    lr              : 0.004039974300410647
    batch_size      : 32
    seq_len         : 10
    alpha           : 4.5
    beta            : 0.3
    move_threshold  : 1.9000000000000001
    gamma           : 2.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
42      42  0.184944                  64                  2             0.3   
37      37  0.150673                  64                  2             0.3   
21      21  0.149608                 128                  2             0.3   
2        2  0.147114                  64                  2             0.4   
6        6  0.143109                 128                  3             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \

[I 2026-06-24 14:11:22,539] A new study created in memory with name: bilstm_attention_AVGO
[I 2026-06-24 14:11:39,946] Trial 0 finished with value: 0.1309886922865703 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: 0.1309886922865703.
[I 2026-06-24 14:11:49,902] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 0 with value: 0.1309886922865703.
[I 2026-06-24 14:12:23,449] Trial 2 finished with value: 0.19598586062706871 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.

...

  Best params saved -> models/AAPL/AAPL_best_params.json

  Best joint score : 0.2146
  Best trial       : 31
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.2
    lr              : 0.0009717023053521539
    batch_size      : 64
    seq_len         : 20
    alpha           : 11.5
    beta            : 0.8
    move_threshold  : 0.8
    gamma           : 2.0

  Top 5 Trials:


[I 2026-06-24 16:28:15,956] A new study created in memory with name: bilstm_attention_TSLA


    number     value  params_hidden_size  params_num_layers  params_dropout  \
31      31  0.214621                  64                  2             0.2   
19      19  0.213241                  64                  2             0.2   
22      22  0.212288                  64                  2             0.2   
37      37  0.209695                  64                  2             0.3   
45      45  0.209627                  64                  2             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
31   0.000972                 64              20          11.5          0.8   
19   0.001010                 64              20          11.0          0.8   
22   0.000974                 64              20          11.5          0.9   
37   0.002660                 32              40           3.0          1.0   
45   0.001584                 64              40           4.0          0.8   

    params_move_threshold  
31                    

[I 2026-06-24 16:28:57,637] Trial 0 finished with value: 0.17419926976116526 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.8, 'gamma': 1.0}. Best is trial 0 with value: 0.17419926976116526.
[I 2026-06-24 16:29:17,239] Trial 1 finished with value: -0.6096118882143896 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 1.4, 'gamma': 0.5}. Best is trial 0 with value: 0.17419926976116526.
[I 2026-06-24 16:29:38,540] Trial 2 finished with value: 0.14273451655035832 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 1.0}. Best is trial 0 w

  Best params saved -> models/TSLA/TSLA_best_params.json

  Best joint score : 0.2001
  Best trial       : 46
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.2
    lr              : 0.0030241450378125476
    batch_size      : 32
    seq_len         : 40
    alpha           : 7.5
    beta            : 0.5
    move_threshold  : 1.2000000000000002
    gamma           : 2.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
46      46  0.200068                 128                  3             0.2   
19      19  0.186691                 128                  3             0.2   
0        0  0.174199                 128                  3             0.3   
44      44  0.171234                 128                  3             0.2   
31      31  0.169980                 128                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
46   0.003024  

In [8]:
# Training 
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import json
import time

# Allow running training WITHOUT re-running Optuna:
# if best_params_all isn't in memory (e.g. kernel restart), cell 6 still runs and
# each symbol's params load from models/{symbol}/{symbol}_best_params.json below.
try:
    best_params_all
except NameError:
    best_params_all = {}

trained_models = {}

def compute_joint_score(price_dir, price_corr,
                          pred_up_pct, std_ratio):
      bias_gap        = abs(pred_up_pct - 0.5)
      std_penalty     = abs(1.0 - std_ratio) * 0.3
      balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0
      return (
          0.40 * price_dir +
          0.25 * max(price_corr,  0.0) +
          std_penalty - bias_gap * 3.0 - balance_penalty
      )


def compute_val_metrics(model, loader, device):
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )

    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])

    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0

    joint_score = (
        0.35 * price_dir               +
        0.25 * max(price_corr,  0.0)   -
        std_penalty                    -
        collapse_penalty
    )

    return joint_score, price_dir, price_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    # if symbol in best_params_all:
    #     best = best_params_all[symbol]
    # else:
    with open(f"models/{symbol}/{symbol}_best_params.json") as f:
        best = json.load(f)["best_params"]
        print(f"  Loaded saved params -> models/{symbol}/{symbol}_best_params.json")
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),
        batch_size=best["batch_size"],
        shuffle=True,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    WARMUP_EPOCHS    = 10
    BASE_ALPHA       = best["alpha"]
    BASE_BETA        = best["beta"]
    BASE_GAMMA       = best.get("gamma", 1.0)
    optimizer        = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler        = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "joint_score": [], "price_dir": [], "price_corr": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | "
          f"{'Pr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)
    
    training_state = type('State', (), {})() 
    
    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = 0.3 + 0.7 * min(1.0, epoch / WARMUP_EPOCHS) 
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = BASE_BETA,
            gamma = BASE_GAMMA
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        joint_score, p_dir, p_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "f"{joint_score:>7.4f} | {p_dir:>6.3f} | "f"{p_corr:>6.3f} | {pred_up_pct:>4.0%} | "f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection with restart ───────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if pred_up_pct > 0.85 or pred_up_pct < 0.15:
                consecutive_collapse = getattr(training_state, 'collapse_ctr', 0) + 1
                training_state.collapse_ctr = consecutive_collapse
                if consecutive_collapse >= 5:
                    print(f"\n  Collapse detected at epoch {epoch} "
                            f"(up%={pred_up_pct:.0%}, std_r={std_ratio:.2f}) — resetting weights")
                    model.apply(lambda m: m.reset_parameters()
                                if hasattr(m, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"]
                    training_state.collapse_ctr = 0
            else:
                training_state.collapse_ctr = 0

        scheduler.step(joint_score)

        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — GOOGL
════════════════════════════════════════════════════════════════════════
  Loaded saved params -> models/GOOGL/GOOGL_best_params.json
  Best params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.30000000000000004, 'lr': 0.004039974300410647, 'batch_size': 32, 'seq_len': 10, 'alpha': 4.5, 'beta': 0.3, 'move_threshold': 1.9000000000000001, 'gamma': 2.0}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |     Pr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 |  0.43603 |  0.50802 |  0.0075 |  0.497 |  0.074 |  69% |  0.08 |  1.1s
         ✓ Saved  (p_dir=0.497, p_corr=0.074, up%=69%, std_r=0.08)
     2 |  0.40426 |  0.50898 |  0.0158 |  0.496 |  0.058 |  76% |  0.14 |  1.0s
         ✓ Saved  (p_dir=0.496, p_corr=0.058, up%=76%, std_r=0.14)
     3 |  0.38659 |  0.53503 |  0.0270 |  0.513 | -0.004 |  52% |  0.24 |  1.2s
  

DIAGNOSTIC

In [9]:
import json
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    if symbol in best_params_all:
        best = best_params_all[symbol]
    else:
        with open(f"models/{symbol}/{symbol}_best_params.json") as f:
            best = json.load(f)["best_params"]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    vol_denom          = data["test_vol_denom"][best["seq_len"]:]   # align: seq target j <-> row j+seq_len
    price_preds_pct    = all_preds[:, 0]   * vol_denom
    price_actuals_pct  = all_actuals[:, 0] * vol_denom

    # ── Metrics ───────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price 5d % Change")

    # ── Save predictions ─────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — GOOGL
═══════════════════════════════════════════════════════

  [Price 5d % Change]
  MAE             : 3.8028%
  RMSE            : 4.8879%
  Pearson r       : 0.0187
  Pred std        : 2.5838  Actual std: 4.1421
  Within ±1%      : 17.1%
  Within ±2%      : 34.4%
  Directional Acc : 47.48%

  Predictions saved: models/GOOGL/GOOGL_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — AVGO
═══════════════════════════════════════════════════════

  [Price 5d % Change]
  MAE             : 4.7837%
  RMSE            : 6.6570%
  Pearson r       : -0.0044
  Pred std        : 3.2403  Actual std: 5.7979
  Within ±1%      : 15.8%
  Within ±2%      : 29.6%
  Directional Acc : 51.93%

  Predictions saved: models/AVGO/AVGO_test_predictions.csv

═══════════════════════════════════════════════════════
  Evaluating — MSFT
═════════════════════════════════════════════

In [10]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  GOOGL — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.5967   std: 4.1432
  Predicted mean : -0.0814   std: 2.5845

  Overall Dir Acc : 47.48%
  When actual UP  : 47.07%  (1075 samples)
  When actual DOWN: 48.03%  (787 samples)

  % predicted UP  : 49.14%
  % actual UP     : 57.73%

  Dir Acc by Move Size:
       <0.5% moves : 50.00%  (212 samples)
      0.5-1% moves : 43.30%  (194 samples)
        1-2% moves : 50.56%  (354 samples)
        2-5% moves : 44.54%  (732 samples)
         >5% moves : 51.08%  (370 samples)

═══════════════════════════════════════════════════════
  AVGO — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.9005   std: 5.7995
  Predicted mean : 0.7096   std: 3.2411

  Overall Dir Acc : 51.93%
  When actual UP  : 59.83%  (1043 samples)
  When actual DOWN: 41.88%  (819 samples)

  % predicted UP  : 59.08%
 